In [1]:
# ============================================================
# D13 — Stage 4 Validation — Branch C
# 0. Imports
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import hashlib
import json
import platform
import re
import sys
import unicodedata

import pandas as pd

In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D13"
DOCUMENT_NAME = "Eurostat — Unemployment rates by country of birth"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

INPUT_REPRESENTATION = (
    "Complete deterministically normalised coordinate-aware structural "
    "Markdown workbook"
)

EXPECTED_SOURCE_SHA256 = (
    "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a"
)

EXPECTED_REFERENCE_SHA256 = (
    "ce230f6b59a15af159b50d831a7adad9b1063c57b3f6e6ef86f1979f8484ab89"
)

EXPECTED_RECORD_COUNT = 75

SELECTED_SHEETS = [
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5"
]

SELECTED_GEOGRAPHIES = [
    "European Union - 27 countries (from 2020)",
    "Belgium",
    "Germany",
    "Spain",
    "Portugal"
]

SELECTED_YEARS = [
    2020,
    2022,
    2024
]

EXPECTED_YEAR_COUNTS = {
    2020: 25,
    2022: 25,
    2024: 25
}

EXPECTED_CATEGORY = "Labour market time series"
EXPECTED_TOPIC = "Unemployment rate by country of birth"
EXPECTED_UNIT = "percent"

EXPECTED_CATEGORY_COUNTS = {
    EXPECTED_CATEGORY: EXPECTED_RECORD_COUNT
}

EXPECTED_FLAG_COUNTS = {
    "d": 10,
    "b": 5,
    "u": 2
}

EXPECTED_FLAGGED_RECORDS = 17

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

# Frozen from final D13 Validation A.
IDENTITY_FIELDS = [
    "Source Location"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period"
]

OUTPUT_DIR = Path("outputs_D13_validation_C_revised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "detailed":
        OUTPUT_DIR / "D13_branch_C_validation_detailed.csv",

    "fully_correct":
        OUTPUT_DIR / "D13_branch_C_fully_correct_records.csv",

    "discrepant":
        OUTPUT_DIR / "D13_branch_C_discrepant_records.csv",

    "missing":
        OUTPUT_DIR / "D13_branch_C_missing_records.csv",

    "unsupported":
        OUTPUT_DIR / "D13_branch_C_unsupported_records.csv",

    "field_validation":
        OUTPUT_DIR / "D13_branch_C_field_validation.csv",

    "category_metrics":
        OUTPUT_DIR / "D13_branch_C_category_metrics.csv",

    "year_metrics":
        OUTPUT_DIR / "D13_branch_C_year_metrics.csv",

    "alignment_issues":
        OUTPUT_DIR / "D13_branch_C_alignment_issues.json",

    "reference_semantics":
        OUTPUT_DIR / "D13_reference_semantics_confirmation.json",

    "summary":
        OUTPUT_DIR / "D13_branch_C_validation_summary.json",

    "metadata":
        OUTPUT_DIR / "D13_branch_C_validation_metadata.json",

    "conclusion":
        OUTPUT_DIR / "D13_branch_C_validation_conclusion.json"
}

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Identity fields:", IDENTITY_FIELDS)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)
print("Output directory:", OUTPUT_DIR)

Document: D13
Branch: C
Expected records: 75
Identity fields: ['Source Location']
Primary correctness fields: ['Category', 'Topic', 'Description', 'Value', 'Unit', 'Reporting Period']
Output directory: outputs_D13_validation_C_revised


In [3]:
# ============================================================
# 2. Upload canonical Stage 1 + Branch C validation inputs
# ============================================================
#
# Required:
#   1) D13_reference_values.csv
#   2) D13_branch_C_structure_check.json
#   3) D13_branch_C_experiment_metadata.json
#   4) D13_branch_C_normalisation_check.json
#   5) D13_branch_C_experiment_summary.json
#
# Required only when Branch C is content-evaluable:
#   6) D13_branch_C_parsed_extraction.json
#
# The experiment summary is checked before the parsed extraction is required.
# If the preserved Branch C response is not content-evaluable, record- and
# field-level metrics must not be forced.

print(
    "Upload:\n"
    "1. D13_reference_values.csv\n"
    "2. D13_branch_C_structure_check.json\n"
    "3. D13_branch_C_experiment_metadata.json\n"
    "4. D13_branch_C_normalisation_check.json\n"
    "5. D13_branch_C_experiment_summary.json\n"
    "6. D13_branch_C_parsed_extraction.json if it was created"
)

uploaded = files.upload()
uploaded_paths = [Path(name) for name in uploaded]

csv_paths = [
    path
    for path in uploaded_paths
    if path.suffix.lower() == ".csv"
]

json_paths = [
    path
    for path in uploaded_paths
    if path.suffix.lower() == ".json"
]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV file: D13_reference_values.csv."
    )

REFERENCE_PATH = csv_paths[0]

EXTRACTION_PATH = None
STRUCTURE_PATH = None
METADATA_PATH = None
NORMALISATION_PATH = None
EXPERIMENT_SUMMARY_PATH = None


def canonical_filename(path):
    return path.name.casefold().replace(" ", "_")


# ------------------------------------------------------------
# First pass: canonical filename patterns
# ------------------------------------------------------------

for path in json_paths:

    filename = canonical_filename(path)

    if "d13_branch_c_parsed_extraction" in filename:
        EXTRACTION_PATH = path
        continue

    if "d13_branch_c_structure_check" in filename:
        STRUCTURE_PATH = path
        continue

    if (
        "d13_branch_c_experiment_metadata" in filename
        and "_pre" not in filename
    ):
        METADATA_PATH = path
        continue

    if (
        "d13_branch_c_normalisation_check" in filename
        or "d13_branch_c_normalization_check" in filename
    ):
        NORMALISATION_PATH = path
        continue

    if "d13_branch_c_experiment_summary" in filename:
        EXPERIMENT_SUMMARY_PATH = path
        continue


# ------------------------------------------------------------
# Second pass: content-based fallback
# ------------------------------------------------------------

for path in json_paths:

    with path.open("r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        EXPERIMENT_SUMMARY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "parsed_extraction_created" in obj
        and "records_evaluable" in obj
        and "validation_status" in obj
    ):
        EXPERIMENT_SUMMARY_PATH = path
        continue

    if (
        NORMALISATION_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and obj.get("parent_branch") == PARENT_BRANCH
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_PATH = path
        continue

    if (
        METADATA_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "source_sha256" in obj
        and "raw_response_sha256" in obj
        and "structure_check_file" in obj
    ):
        METADATA_PATH = path
        continue

    if (
        STRUCTURE_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structure_valid" in obj
        and "records_evaluable" in obj
        and "validation_status" not in obj
    ):
        STRUCTURE_PATH = path
        continue

    if (
        EXTRACTION_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        EXTRACTION_PATH = path
        continue


for label, path in {
    "structure check": STRUCTURE_PATH,
    "experiment metadata": METADATA_PATH,
    "normalisation check": NORMALISATION_PATH,
    "experiment summary": EXPERIMENT_SUMMARY_PATH,
}.items():

    if path is None:
        raise ValueError(
            f"Could not identify required {label} file."
        )


with EXPERIMENT_SUMMARY_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    experiment_summary_payload = json.load(f)


content_evaluable = bool(
    experiment_summary_payload.get("records_evaluable")
    and experiment_summary_payload.get("parsed_extraction_created")
)


if content_evaluable and EXTRACTION_PATH is None:
    raise ValueError(
        "This D13 Branch C execution is content-evaluable, so "
        "D13_branch_C_parsed_extraction.json is required."
    )

if not content_evaluable:
    raise ValueError(
        "This D13 Branch C execution is not content-evaluable. "
        "Do not calculate record- or field-level metrics. "
        "Use a non-evaluable validation pathway like D9."
    )


required_paths = [
    STRUCTURE_PATH,
    METADATA_PATH,
    NORMALISATION_PATH,
    EXPERIMENT_SUMMARY_PATH,
    EXTRACTION_PATH,
]

if len({str(path) for path in required_paths}) != len(required_paths):
    raise ValueError(
        "The same JSON file was assigned to more than one "
        "Branch C artefact role."
    )


print("\nCanonical Branch C validation inputs resolved:")
print("Reference:", REFERENCE_PATH.name)
print("Parsed extraction:", EXTRACTION_PATH.name)
print("Structure check:", STRUCTURE_PATH.name)
print("Experiment metadata:", METADATA_PATH.name)
print("Normalisation check:", NORMALISATION_PATH.name)
print("Experiment summary:", EXPERIMENT_SUMMARY_PATH.name)
print("Content evaluable:", content_evaluable)

Upload:
1. D13_reference_values.csv
2. D13_branch_C_structure_check.json
3. D13_branch_C_experiment_metadata.json
4. D13_branch_C_normalisation_check.json
5. D13_branch_C_experiment_summary.json
6. D13_branch_C_parsed_extraction.json if it was created


Saving D13_branch_C_structure_check.json to D13_branch_C_structure_check.json
Saving D13_branch_C_parsed_extraction.json to D13_branch_C_parsed_extraction.json
Saving D13_branch_C_normalisation_check.json to D13_branch_C_normalisation_check.json
Saving D13_branch_C_experiment_summary.json to D13_branch_C_experiment_summary.json
Saving D13_branch_C_experiment_metadata.json to D13_branch_C_experiment_metadata.json
Saving D13_reference_values.csv to D13_reference_values.csv

Canonical Branch C validation inputs resolved:
Reference: D13_reference_values.csv
Parsed extraction: D13_branch_C_parsed_extraction.json
Structure check: D13_branch_C_structure_check.json
Experiment metadata: D13_branch_C_experiment_metadata.json
Normalisation check: D13_branch_C_normalisation_check.json
Experiment summary: D13_branch_C_experiment_summary.json
Content evaluable: True


In [4]:
# ============================================================
# 3. Load and fingerprint inputs
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
STRUCTURE_SHA256 = sha256_file(STRUCTURE_PATH)
METADATA_SHA256 = sha256_file(METADATA_PATH)
NORMALISATION_SHA256 = sha256_file(NORMALISATION_PATH)
EXPERIMENT_SUMMARY_SHA256 = sha256_file(
    EXPERIMENT_SUMMARY_PATH
)


reference_df = pd.read_csv(
    REFERENCE_PATH,
    encoding="utf-8-sig",
    keep_default_na=False
)

with EXTRACTION_PATH.open("r", encoding="utf-8") as f:
    extraction_payload = json.load(f)

with STRUCTURE_PATH.open("r", encoding="utf-8") as f:
    structure_payload = json.load(f)

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata_payload = json.load(f)

with NORMALISATION_PATH.open("r", encoding="utf-8") as f:
    normalisation_payload = json.load(f)


extracted_records = extraction_payload.get("records", [])

extracted_df = pd.DataFrame(
    extracted_records
)


print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Structure-check SHA-256:", STRUCTURE_SHA256)
print("Experiment-metadata SHA-256:", METADATA_SHA256)
print("Normalisation-integrity SHA-256:", NORMALISATION_SHA256)
print("Experiment-summary SHA-256:", EXPERIMENT_SUMMARY_SHA256)
print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_records))

Reference SHA-256: ce230f6b59a15af159b50d831a7adad9b1063c57b3f6e6ef86f1979f8484ab89
Extraction SHA-256: 47b0f2a6476ca8259d654e10bf141d53a364f7c5a62c6b1e2f8ff9fde7d7a8f5
Structure-check SHA-256: 118c51ff5c8f78bc1435733878de59ce73c10f4e77933260072a9a03a5a12774
Experiment-metadata SHA-256: af5af30abac0d20e5c890435323faf9574c8fcada53f19dffebb4a7bffdb0298
Normalisation-integrity SHA-256: 186a219fe6f26b3e285cf8bf13f17bd162f7d0ff444384ec75d01edc51f532f4
Experiment-summary SHA-256: 766a7369dc5481012f1e3d18cc6779a375048f5452de1df45660b084a27e5ee3
Reference records: 75
Extracted records: 75


In [5]:
# ============================================================
# 4. Confirm fixed Stage 1 D13 reference semantics
# ============================================================

def normalise_text(value):
    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    text = (
        text.replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("—", "-")
        .replace("–", "-")
        .replace("’", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def canonical_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    if re.fullmatch(r"\d{4}", text):
        return int(text)

    return text


def canonical_source_location(value):
    text = normalise_text(value)

    if text is None:
        return None

    match = re.fullmatch(
        r"(?i)sheet\s+(\d+)\s*,\s*cell\s+([A-Z]+)(\d+)",
        text
    )

    if not match:
        return text

    return (
        f"Sheet {int(match.group(1))}, "
        f"cell {match.group(2).upper()}{int(match.group(3))}"
    )


reference_schema_exact = (
    reference_df.columns.tolist() == EXPECTED_FIELDS
)

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
    if "Category" in reference_df.columns
    else {}
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

reference_category_constant = bool(
    (reference_df["Category"] == EXPECTED_CATEGORY).all()
)

reference_topic_constant = bool(
    (reference_df["Topic"] == EXPECTED_TOPIC).all()
)

reference_unit_constant = bool(
    (reference_df["Unit"] == EXPECTED_UNIT).all()
)

reference_values_numeric_series = pd.to_numeric(
    reference_df["Value"],
    errors="coerce"
)

reference_values_numeric = bool(
    reference_values_numeric_series.notna().all()
)

reference_values_in_percent_range = bool(
    (
        (reference_values_numeric_series >= 0)
        & (reference_values_numeric_series <= 100)
    ).all()
)

reference_periods = reference_df[
    "Reporting Period"
].apply(canonical_period)

reference_year_counts = dict(
    Counter(reference_periods)
)

reference_year_counts_valid = (
    reference_year_counts == EXPECTED_YEAR_COUNTS
)

reference_source_locations = reference_df[
    "Source Location"
].apply(canonical_source_location)

reference_source_location_format_valid = bool(
    reference_source_locations
    .astype(str)
    .str.fullmatch(
        r"Sheet [1-5], cell [A-Z]+\d+"
    )
    .all()
)

reference_identity_unique = (
    reference_source_locations.nunique()
    == EXPECTED_RECORD_COUNT
)

description_required_labels = [
    "Geography:",
    "Sex:",
    "Age class:",
    "Country/region of birth:"
]

reference_description_structure_valid = bool(
    reference_df["Description"].apply(
        lambda text: all(
            label in str(text)
            for label in description_required_labels
        )
    ).all()
)

flag_pattern = re.compile(
    r";\s*Statistical flag:\s*([^;]+)\s*$"
)

observed_reference_flags = []

for description in reference_df["Description"].astype(str):
    match = flag_pattern.search(description)

    if match:
        observed_reference_flags.append(
            match.group(1).strip()
        )

reference_flag_counts = dict(
    Counter(observed_reference_flags)
)

reference_flag_count_valid = (
    len(observed_reference_flags)
    == EXPECTED_FLAGGED_RECORDS
)

reference_flag_values_valid = (
    reference_flag_counts == EXPECTED_FLAG_COUNTS
)

reference_sha_matches_stage_1 = (
    REFERENCE_SHA256 == EXPECTED_REFERENCE_SHA256
)

reference_semantic_checks = {
    "reference_schema_exact":
        bool(reference_schema_exact),
    "reference_record_count_valid":
        bool(reference_record_count_valid),
    "reference_category_counts_valid":
        bool(reference_category_counts_valid),
    "reference_category_constant":
        bool(reference_category_constant),
    "reference_topic_constant":
        bool(reference_topic_constant),
    "reference_unit_constant":
        bool(reference_unit_constant),
    "reference_values_numeric":
        bool(reference_values_numeric),
    "reference_values_in_percent_range":
        bool(reference_values_in_percent_range),
    "reference_year_counts_valid":
        bool(reference_year_counts_valid),
    "reference_source_location_format_valid":
        bool(reference_source_location_format_valid),
    "reference_identity_unique":
        bool(reference_identity_unique),
    "reference_description_structure_valid":
        bool(reference_description_structure_valid),
    "reference_flag_count_valid":
        bool(reference_flag_count_valid),
    "reference_flag_values_valid":
        bool(reference_flag_values_valid),
    "reference_sha_matches_stage_1":
        bool(reference_sha_matches_stage_1)
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

REFERENCE_SEMANTICS_CONFIRMATION = {
    "document_id":
        DOCUMENT_ID,
    "reference_semantics_valid":
        bool(reference_semantics_valid),
    "checks":
        reference_semantic_checks,
    "observed_flag_counts":
        reference_flag_counts,
    "observed_year_counts":
        reference_year_counts
}

PATHS["reference_semantics"].write_text(
    json.dumps(
        REFERENCE_SEMANTICS_CONFIRMATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        REFERENCE_SEMANTICS_CONFIRMATION,
        ensure_ascii=False,
        indent=2
    )
)

if not reference_semantics_valid:
    raise AssertionError(
        "D13 Stage 1 reference semantics are not valid."
    )

{
  "document_id": "D13",
  "reference_semantics_valid": true,
  "checks": {
    "reference_schema_exact": true,
    "reference_record_count_valid": true,
    "reference_category_counts_valid": true,
    "reference_category_constant": true,
    "reference_topic_constant": true,
    "reference_unit_constant": true,
    "reference_values_numeric": true,
    "reference_values_in_percent_range": true,
    "reference_year_counts_valid": true,
    "reference_source_location_format_valid": true,
    "reference_identity_unique": true,
    "reference_description_structure_valid": true,
    "reference_flag_count_valid": true,
    "reference_flag_values_valid": true,
    "reference_sha_matches_stage_1": true
  },
  "observed_flag_counts": {
    "b": 5,
    "d": 10,
    "u": 2
  },
  "observed_year_counts": {
    "2020": 25,
    "2022": 25,
    "2024": 25
  }
}


In [6]:
# ============================================================
# 5. Confirm Branch C provenance, schema and normalisation integrity
# ============================================================

top_level_object_valid = isinstance(
    extraction_payload,
    dict
)

document_id_correct = (
    extraction_payload.get("document_id")
    == DOCUMENT_ID
)

branch_correct = (
    extraction_payload.get("branch")
    == BRANCH
)

records_is_list = isinstance(
    extraction_payload.get("records"),
    list
)

record_schema_valid = True
field_types_valid = True
schema_issue_rows = []

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


if records_is_list:

    for i, record in enumerate(
        extracted_records
    ):

        if not isinstance(record, dict):

            record_schema_valid = False

            schema_issue_rows.append({
                "record_index": i,
                "issue": "record_not_object"
            })

            continue


        if list(record.keys()) != EXPECTED_FIELDS:

            record_schema_valid = False

            schema_issue_rows.append({
                "record_index": i,
                "issue": "field_names_or_order",
                "observed_fields": list(record.keys())
            })


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(field)

            if (
                value is not None
                and not isinstance(value, str)
            ):

                field_types_valid = False

                schema_issue_rows.append({
                    "record_index": i,
                    "issue": "field_type",
                    "field": field,
                    "observed_type":
                        type(value).__name__
                })


        value = record.get("Value")

        if (
            value is not None
            and (
                isinstance(value, bool)
                or not isinstance(
                    value,
                    (int, float)
                )
            )
        ):

            field_types_valid = False

            schema_issue_rows.append({
                "record_index": i,
                "issue": "field_type",
                "field": "Value",
                "observed_type":
                    type(value).__name__
            })


else:

    record_schema_valid = False
    field_types_valid = False


branch_C_structure_valid = bool(
    structure_payload.get(
        "structure_valid"
    )
)


schema_validity = all([
    top_level_object_valid,
    document_id_correct,
    branch_correct,
    records_is_list,
    record_schema_valid,
    field_types_valid,
    branch_C_structure_valid
])


metadata_parsed_hash = (
    metadata_payload.get(
        "parsed_extraction_sha256"
    )
)

metadata_source_hash = (
    metadata_payload.get(
        "source_sha256"
    )
)

parsed_extraction_hash_matches_metadata = (
    metadata_parsed_hash
    == EXTRACTION_SHA256
)

source_hash_matches_stage_1 = (
    metadata_source_hash
    == EXPECTED_SOURCE_SHA256
)

metadata_document_id_correct = (
    metadata_payload.get("document_id")
    == DOCUMENT_ID
)

metadata_branch_correct = (
    metadata_payload.get("branch")
    == BRANCH
)


parent_equivalence_passed = bool(
    normalisation_payload.get(
        "parent_equivalence_passed",
        False
    )
)

normalisation_integrity_passed = bool(
    normalisation_payload.get(
        "normalisation_integrity_passed",
        False
    )
)

normalisation_document_id_correct = (
    normalisation_payload.get("document_id")
    == DOCUMENT_ID
)

normalisation_branch_correct = (
    normalisation_payload.get("branch")
    == BRANCH
)

normalisation_parent_branch_correct = (
    normalisation_payload.get("parent_branch")
    == PARENT_BRANCH
)


representation_integrity = {
    "parent_branch":
        normalisation_payload.get(
            "parent_branch"
        ),

    "parent_equivalence_passed":
        parent_equivalence_passed,

    "normalisation_integrity_passed":
        normalisation_integrity_passed,

    "expected_worksheet_count":
        normalisation_payload.get(
            "expected_worksheet_count"
        ),

    "parent_worksheet_count":
        normalisation_payload.get(
            "parent_worksheet_count"
        ),

    "branch_C_worksheet_count":
        normalisation_payload.get(
            "branch_C_worksheet_count"
        ),

    "worksheet_order_preserved":
        normalisation_payload.get(
            "worksheet_order_preserved"
        ),

    "expected_non_empty_cell_count":
        normalisation_payload.get(
            "expected_non_empty_cell_count"
        ),

    "parent_represented_cell_count":
        normalisation_payload.get(
            "parent_represented_cell_count"
        ),

    "branch_C_represented_cell_count":
        normalisation_payload.get(
            "branch_C_represented_cell_count"
        ),

    "cell_count_preserved":
        normalisation_payload.get(
            "cell_count_preserved"
        ),

    "cell_identity_type_and_order_preserved":
        normalisation_payload.get(
            "cell_identity_type_and_order_preserved"
        ),

    "all_source_cells_preserved":
        normalisation_payload.get(
            "all_source_cells_preserved"
        ),

    "string_values_correctly_normalised":
        normalisation_payload.get(
            "string_values_correctly_normalised"
        ),

    "normalised_string_cell_count":
        normalisation_payload.get(
            "normalised_string_cell_count"
        ),

    "non_string_values_preserved":
        normalisation_payload.get(
            "non_string_values_preserved"
        ),

    "selected_scope_value_cells_preserved":
        normalisation_payload.get(
            "selected_scope_value_cells_preserved"
        ),

    "selected_scope_flag_cells_preserved_when_nonempty":
        normalisation_payload.get(
            "selected_scope_flag_cells_preserved_when_nonempty"
        ),

    "deterministic_representation_verified":
        normalisation_payload.get(
            "deterministic_representation_verified"
        ),

    "complete_16_sheet_representation_retained":
        normalisation_payload.get(
            "complete_16_sheet_representation_retained"
        ),

    "complete_source_cell_population_retained":
        normalisation_payload.get(
            "complete_source_cell_population_retained"
        ),

    "worksheet_filtering_applied":
        normalisation_payload.get(
            "worksheet_filtering_applied"
        ),

    "row_filtering_applied":
        normalisation_payload.get(
            "row_filtering_applied"
        ),

    "column_filtering_applied":
        normalisation_payload.get(
            "column_filtering_applied"
        ),

    "selected_scope_filtering_applied":
        normalisation_payload.get(
            "selected_scope_filtering_applied"
        ),

    "cell_reordering_applied":
        normalisation_payload.get(
            "cell_reordering_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_payload.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_payload.get(
            "semantic_rewriting_applied"
        ),

    "statistical_flag_reconstruction_applied":
        normalisation_payload.get(
            "statistical_flag_reconstruction_applied"
        ),

    "unit_conversion_applied":
        normalisation_payload.get(
            "unit_conversion_applied"
        ),

    "numeric_calculation_applied":
        normalisation_payload.get(
            "numeric_calculation_applied"
        ),

    "numeric_rescaling_applied":
        normalisation_payload.get(
            "numeric_rescaling_applied"
        ),

    "numeric_rounding_applied":
        normalisation_payload.get(
            "numeric_rounding_applied"
        ),

    "manual_reconstruction_applied":
        normalisation_payload.get(
            "manual_reconstruction_applied"
        ),

    "manual_correction_applied":
        normalisation_payload.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_payload.get(
            "reference_values_used_for_transformation"
        )
}


input_provenance = {
    "reference_file":
        REFERENCE_PATH.name,

    "reference_sha256":
        REFERENCE_SHA256,

    "reference_sha_matches_stage_1":
        reference_sha_matches_stage_1,

    "parsed_extraction_file":
        EXTRACTION_PATH.name,

    "parsed_extraction_sha256":
        EXTRACTION_SHA256,

    "structure_check_file":
        STRUCTURE_PATH.name,

    "structure_check_sha256":
        STRUCTURE_SHA256,

    "experiment_metadata_file":
        METADATA_PATH.name,

    "experiment_metadata_sha256":
        METADATA_SHA256,

    "normalisation_integrity_file":
        NORMALISATION_PATH.name,

    "normalisation_integrity_sha256":
        NORMALISATION_SHA256,

    "experiment_summary_file":
        EXPERIMENT_SUMMARY_PATH.name,

    "experiment_summary_sha256":
        EXPERIMENT_SUMMARY_SHA256,

    "branch_C_structure_valid":
        branch_C_structure_valid,

    "parsed_extraction_hash_matches_metadata":
        parsed_extraction_hash_matches_metadata,

    "source_hash_matches_stage_1":
        source_hash_matches_stage_1,

    "parent_B_equivalence_passed":
        parent_equivalence_passed,

    "normalisation_integrity_passed":
        normalisation_integrity_passed,

    "metadata_document_id_correct":
        metadata_document_id_correct,

    "metadata_branch_correct":
        metadata_branch_correct,

    "normalisation_document_id_correct":
        normalisation_document_id_correct,

    "normalisation_branch_correct":
        normalisation_branch_correct,

    "normalisation_parent_branch_correct":
        normalisation_parent_branch_correct
}


print("Schema validity:", schema_validity)
print(
    "Branch C structure valid:",
    branch_C_structure_valid
)
print(
    "Parent B equivalence passed:",
    parent_equivalence_passed
)
print(
    "Normalisation integrity passed:",
    normalisation_integrity_passed
)
print(
    json.dumps(
        input_provenance,
        ensure_ascii=False,
        indent=2
    )
)

Schema validity: True
Branch C structure valid: True
Parent B equivalence passed: True
Normalisation integrity passed: True
{
  "reference_file": "D13_reference_values.csv",
  "reference_sha256": "ce230f6b59a15af159b50d831a7adad9b1063c57b3f6e6ef86f1979f8484ab89",
  "reference_sha_matches_stage_1": true,
  "parsed_extraction_file": "D13_branch_C_parsed_extraction.json",
  "parsed_extraction_sha256": "47b0f2a6476ca8259d654e10bf141d53a364f7c5a62c6b1e2f8ff9fde7d7a8f5",
  "structure_check_file": "D13_branch_C_structure_check.json",
  "structure_check_sha256": "118c51ff5c8f78bc1435733878de59ce73c10f4e77933260072a9a03a5a12774",
  "experiment_metadata_file": "D13_branch_C_experiment_metadata.json",
  "experiment_metadata_sha256": "af5af30abac0d20e5c890435323faf9574c8fcada53f19dffebb4a7bffdb0298",
  "normalisation_integrity_file": "D13_branch_C_normalisation_check.json",
  "normalisation_integrity_sha256": "186a219fe6f26b3e285cf8bf13f17bd162f7d0ff444384ec75d01edc51f532f4",
  "experiment_summary

In [7]:
# ============================================================
# 6. Comparison-only canonicalisation
# ============================================================

def parse_numeric(value):
    if value is None or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    text = normalise_text(value)

    if text is None:
        return None

    text = text.replace(",", "")

    if re.fullmatch(
        r"[-+]?\d+(?:\.\d+)?",
        text
    ):
        return float(text)

    return None


def compare_field(
    field,
    reference_value,
    extracted_value
):
    if field == "Value":
        reference_numeric = parse_numeric(
            reference_value
        )

        extracted_numeric = parse_numeric(
            extracted_value
        )

        return (
            reference_numeric is not None
            and extracted_numeric is not None
            and reference_numeric
            == extracted_numeric
        )

    if field == "Reporting Period":
        return (
            canonical_period(reference_value)
            == canonical_period(extracted_value)
        )

    if field == "Source Location":
        return (
            canonical_source_location(reference_value)
            == canonical_source_location(extracted_value)
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )

In [8]:
# ============================================================
# 7. Deterministic one-to-one alignment by physical source cell
# ============================================================

def build_identity_index(records, dataset_name):
    index = {}
    duplicate_groups = []

    for record_index, record in enumerate(records):

        if not isinstance(record, dict):
            continue

        identity = canonical_source_location(
            record.get("Source Location")
        )

        if identity in index:
            duplicate_groups.append({
                "dataset":
                    dataset_name,
                "identity":
                    identity,
                "first_record_index":
                    index[identity]["record_index"],
                "duplicate_record_index":
                    record_index
            })

        else:
            index[identity] = {
                "record_index":
                    record_index,
                "record":
                    record
            }

    return index, duplicate_groups


reference_records = reference_df.to_dict(
    "records"
)

reference_index, reference_duplicates = (
    build_identity_index(
        reference_records,
        "Reference"
    )
)

extraction_index, extraction_duplicates = (
    build_identity_index(
        extracted_records,
        "Extraction"
    )
)

alignment_issues = (
    reference_duplicates
    + extraction_duplicates
)

PATHS["alignment_issues"].write_text(
    json.dumps(
        alignment_issues,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Reference duplicate identities:",
    len(reference_duplicates)
)
print(
    "Extraction duplicate identities:",
    len(extraction_duplicates)
)

if reference_duplicates:
    raise AssertionError(
        "Reference source-cell identity is not unique."
    )

Reference duplicate identities: 0
Extraction duplicate identities: 0


In [9]:
# ============================================================
# 8. Match records and compare fields
# ============================================================

detailed_rows = []
discrepant_rows = []
missing_rows = []
unsupported_rows = []

all_identities = sorted(
    set(reference_index.keys())
    | set(extraction_index.keys()),
    key=lambda x: (x is None, str(x))
)

for identity in all_identities:
    ref_entry = reference_index.get(identity)
    ext_entry = extraction_index.get(identity)

    if (
        ref_entry is not None
        and ext_entry is None
    ):
        row = ref_entry["record"].copy()
        row[
            "Reference Record Index"
        ] = ref_entry["record_index"]

        missing_rows.append(row)
        continue

    if (
        ext_entry is not None
        and ref_entry is None
    ):
        row = ext_entry["record"].copy()
        row[
            "Extracted Record Index"
        ] = ext_entry["record_index"]

        unsupported_rows.append(row)
        continue

    ref_record = ref_entry["record"]
    ext_record = ext_entry["record"]

    field_results = {
        field: compare_field(
            field,
            ref_record.get(field),
            ext_record.get(field)
        )
        for field in EXPECTED_FIELDS
    }

    primary_correct = all(
        field_results[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    # Source Location is the identity field, so agreement is guaranteed
    # for aligned records and is not double-counted as primary correctness.
    identity_correct = all(
        field_results[field]
        for field in IDENTITY_FIELDS
    )

    fully_correct = (
        primary_correct
        and identity_correct
    )

    detail = {
        "Reference Record Index":
            ref_entry["record_index"],
        "Extracted Record Index":
            ext_entry["record_index"],
        "Identity Source Location":
            identity,
        "Fully Correct":
            fully_correct,
        "Primary Correct":
            primary_correct
    }

    for field in EXPECTED_FIELDS:
        detail[
            f"Reference {field}"
        ] = ref_record.get(field)

        detail[
            f"Extracted {field}"
        ] = ext_record.get(field)

        detail[
            f"{field} Correct"
        ] = field_results[field]

    detailed_rows.append(detail)

    if not fully_correct:
        discrepant_rows.append(
            detail.copy()
        )


detailed_df = pd.DataFrame(
    detailed_rows
)

discrepant_df = pd.DataFrame(
    discrepant_rows
)

missing_df = pd.DataFrame(
    missing_rows
)

unsupported_df = pd.DataFrame(
    unsupported_rows
)

print("Aligned:", len(detailed_df))
print("Discrepant:", len(discrepant_df))
print("Missing:", len(missing_df))
print(
    "Unsupported/unmatched:",
    len(unsupported_df)
)

Aligned: 75
Discrepant: 0
Missing: 0
Unsupported/unmatched: 0


In [10]:
# ============================================================
# 9. Calculate common validation metrics
# ============================================================

reference_count = len(reference_df)
extracted_count = len(extracted_records)
aligned_count = len(detailed_df)

fully_correct_count = (
    int(
        detailed_df["Fully Correct"].sum()
    )
    if aligned_count
    else 0
)

discrepant_count = len(discrepant_df)
missing_count = len(missing_df)
unsupported_count = len(unsupported_df)

completeness = (
    aligned_count / reference_count
    if reference_count
    else None
)

record_precision_exact = (
    fully_correct_count / extracted_count
    if extracted_count
    else 0.0
)

record_recall_exact = (
    fully_correct_count / reference_count
    if reference_count
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)


primary_field_accuracy = {}

for field in PRIMARY_CORRECTNESS_FIELDS:
    if aligned_count:
        primary_field_accuracy[field] = float(
            detailed_df[
                f"{field} Correct"
            ].mean()
        )
    else:
        primary_field_accuracy[field] = None


valid_primary_values = [
    value
    for value in primary_field_accuracy.values()
    if value is not None
]

overall_primary_field_accuracy = (
    sum(valid_primary_values)
    / len(valid_primary_values)
    if valid_primary_values
    else None
)


# ------------------------------------------------------------
# Category metrics
# ------------------------------------------------------------

category_metrics = {}

for category in sorted(
    set(reference_df["Category"].astype(str))
):
    ref_category_count = int(
        (
            reference_df["Category"]
            == category
        ).sum()
    )

    ext_category_count = sum(
        1
        for record in extracted_records
        if isinstance(record, dict)
        and record.get("Category")
        == category
    )

    aligned_category = (
        detailed_df[
            detailed_df[
                "Reference Category"
            ] == category
        ]
        if aligned_count
        else pd.DataFrame()
    )

    aligned_category_count = len(
        aligned_category
    )

    fully_correct_category = (
        int(
            aligned_category[
                "Fully Correct"
            ].sum()
        )
        if aligned_category_count
        else 0
    )

    p = (
        fully_correct_category
        / ext_category_count
        if ext_category_count
        else 0.0
    )

    r = (
        fully_correct_category
        / ref_category_count
        if ref_category_count
        else 0.0
    )

    f1 = (
        2 * p * r / (p + r)
        if p + r
        else 0.0
    )

    category_metrics[category] = {
        "expected_records":
            ref_category_count,
        "extracted_records":
            ext_category_count,
        "aligned_records":
            aligned_category_count,
        "fully_correct_records":
            fully_correct_category,
        "discrepant_records":
            aligned_category_count
            - fully_correct_category,
        "completeness":
            aligned_category_count
            / ref_category_count
            if ref_category_count
            else None,
        "record_precision_exact":
            p,
        "record_recall_exact":
            r,
        "record_f1_exact":
            f1
    }


# ------------------------------------------------------------
# Reporting-period diagnostics
# ------------------------------------------------------------

year_metrics = {}

for year in SELECTED_YEARS:
    ref_year_count = int(
        (
            reference_df[
                "Reporting Period"
            ].apply(canonical_period)
            == year
        ).sum()
    )

    ext_year_count = sum(
        1
        for record in extracted_records
        if isinstance(record, dict)
        and canonical_period(
            record.get("Reporting Period")
        ) == year
    )

    aligned_year = (
        detailed_df[
            detailed_df[
                "Reference Reporting Period"
            ].apply(canonical_period)
            == year
        ]
        if aligned_count
        else pd.DataFrame()
    )

    aligned_year_count = len(aligned_year)

    fully_correct_year = (
        int(
            aligned_year[
                "Fully Correct"
            ].sum()
        )
        if aligned_year_count
        else 0
    )

    year_metrics[str(year)] = {
        "expected_records":
            ref_year_count,
        "extracted_records":
            ext_year_count,
        "aligned_records":
            aligned_year_count,
        "fully_correct_records":
            fully_correct_year,
        "discrepant_records":
            aligned_year_count
            - fully_correct_year
    }

In [11]:
# ============================================================
# 10. Create final Branch C validation summary
# ============================================================

comparison_rules = {
    "raw_extraction_modified":
        False,

    "manual_correction_applied":
        False,

    "comparison_normalisation_scope":
        "Comparison copies only",

    "identity_fields":
        IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "value":
        (
            "Exact represented numeric equality after deterministic "
            "parsing; no tolerance, rounding, interpolation, "
            "rescaling or conversion."
        ),

    "category_topic_unit":
        (
            "Conservative normalised exact agreement with the fixed "
            "Stage 1 semantic values."
        ),

    "description":
        (
            "Normalised exact agreement. The D13 prompt defines the "
            "Description template explicitly, including source-grounded "
            "worksheet dimensions and the adjacent statistical flag "
            "when present."
        ),

    "reporting_period":
        "Canonical exact year agreement.",

    "source_location":
        (
            "Physical workbook value-cell coordinate defines D13 "
            "observation identity and is therefore used for alignment, "
            "not double-counted as a primary correctness field."
        ),

    "d13_equivalence_rules_status":
        (
            "Final D13 Validation A identity and comparison rules reused "
            "unchanged for Branch C. No Branch-C-specific semantic "
            "equivalence or performance-driven matching rule was added."
        ),

    "equivalence_rules_frozen":
        True
}


matching_rules = {
    "identity_fields":
        IDENTITY_FIELDS,

    "one_to_one_assignment":
        (
            "Unique deterministic canonical workbook Source Location "
            "(physical value-cell coordinate)."
        ),

    "category_used_for_alignment":
        False,

    "topic_used_for_alignment":
        False,

    "description_used_for_alignment":
        False,

    "value_used_for_alignment":
        False,

    "unit_used_for_alignment":
        False,

    "reporting_period_used_for_alignment":
        False,

    "matching_rules_frozen_from_branch_A":
        True
}


content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "reference_year_counts_valid":
        bool(reference_year_counts_valid),

    "reference_flag_count_valid":
        bool(reference_flag_count_valid),

    "reference_flag_values_valid":
        bool(reference_flag_values_valid),

    "extraction_record_count_valid":
        bool(
            extracted_count
            == EXPECTED_RECORD_COUNT
        ),

    "extraction_category_counts_valid":
        dict(
            Counter(
                record.get("Category")
                for record in extracted_records
                if isinstance(record, dict)
            )
        )
        == EXPECTED_CATEGORY_COUNTS,

    "branch_C_scope_complete":
        structure_payload.get(
            "scope_complete"
        ),

    "branch_C_content_diagnostics":
        structure_payload.get(
            "content_diagnostics"
        ),

    "reference_identity_unique":
        bool(reference_identity_unique),

    "extraction_duplicate_identity_count":
        int(
            len(extraction_duplicates)
        ),

    "ambiguous_identity_group_count":
        int(
            len(alignment_issues)
        )
}


schema_diagnostics = {
    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_correct":
        bool(document_id_correct),

    "branch_correct":
        bool(branch_correct),

    "records_is_list":
        bool(records_is_list),

    "record_schema_valid":
        bool(record_schema_valid),

    "field_types_valid":
        bool(field_types_valid),

    "branch_C_structure_valid":
        bool(branch_C_structure_valid),

    "schema_validity":
        bool(schema_validity)
}


VALIDATION_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "reference_records":
        reference_count,

    "extracted_records":
        extracted_count,

    "aligned_records":
        aligned_count,

    "fully_correct_records":
        fully_correct_count,

    "discrepant_records":
        discrepant_count,

    "missing_records":
        missing_count,

    "unsupported_extracted_records":
        unsupported_count,

    "completeness":
        completeness,

    "missing_rate": (
        missing_count
        / reference_count
        if reference_count
        else None
    ),

    "record_precision_exact":
        record_precision_exact,

    "record_recall_exact":
        record_recall_exact,

    "record_f1_exact":
        record_f1_exact,

    "unsupported_rate": (
        unsupported_count
        / extracted_count
        if extracted_count
        else None
    ),

    "discrepancy_rate_among_aligned": (
        discrepant_count
        / aligned_count
        if aligned_count
        else None
    ),

    "overall_primary_field_accuracy":
        overall_primary_field_accuracy,

    "primary_field_accuracy":
        primary_field_accuracy,

    "schema_validity":
        bool(schema_validity),

    "schema_diagnostics":
        schema_diagnostics,

    "content_diagnostics":
        content_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_rules":
        matching_rules,

    "comparison_rules":
        comparison_rules,

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(reference_semantics_valid),

        "checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False
    },

    "category_metrics":
        category_metrics,

    "year_metrics":
        year_metrics,

    "input_provenance":
        input_provenance,

    "comparison_rules_frozen_from_branch_A":
        True,

    "validation_timestamp":
        datetime.now(
            timezone.utc
        ).isoformat()
}


print(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D13",
  "document_name": "Eurostat — Unemployment rates by country of birth",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised coordinate-aware structural Markdown workbook",
  "reference_records": 75,
  "extracted_records": 75,
  "aligned_records": 75,
  "fully_correct_records": 75,
  "discrepant_records": 0,
  "missing_records": 0,
  "unsupported_extracted_records": 0,
  "completeness": 1.0,
  "missing_rate": 0.0,
  "record_precision_exact": 1.0,
  "record_recall_exact": 1.0,
  "record_f1_exact": 1.0,
  "unsupported_rate": 0.0,
  "discrepancy_rate_among_aligned": 0.0,
  "overall_primary_field_accuracy": 1.0,
  "primary_field_accuracy": {
    "Category": 1.0,
    "Topic": 1.0,
    "Description": 1.0,
    "Value": 1.0,
    "Unit": 1.0,
    "Reporting Period": 1.0
  },
  "schema_validity": true,
  "schema_diagnostics": {
    "top_level_object_valid": true,
    "document_id_correct": true

In [12]:
# ============================================================
# 11. Export Validation C outputs and run final consistency checks
# ============================================================

fully_correct_df = (
    detailed_df[
        detailed_df["Fully Correct"]
    ].copy()
)

field_validation_df = pd.DataFrame([
    {
        "Field":
            field,

        "Role": (
            "Identity"
            if field in IDENTITY_FIELDS
            else "Primary correctness"
        ),

        "Correct":
            int(
                detailed_df[
                    f"{field} Correct"
                ].sum()
            )
            if aligned_count
            else 0,

        "Compared":
            int(aligned_count),

        "Accuracy": (
            float(
                detailed_df[
                    f"{field} Correct"
                ].mean()
            )
            if aligned_count
            else 0.0
        )
    }
    for field in EXPECTED_FIELDS
])


category_metrics_df = pd.DataFrame([
    {
        "Category":
            category,
        **metrics
    }
    for category, metrics
    in category_metrics.items()
])


year_metrics_df = pd.DataFrame([
    {
        "Reporting Period":
            year,
        **metrics
    }
    for year, metrics
    in year_metrics.items()
])


VALIDATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "validation_type":
        "Post-extraction reference-value agreement",

    "identity_fields":
        IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "raw_extraction_modified":
        False,

    "manual_correction_applied":
        False,

    "comparison_normalisation_scope":
        "Comparison copies only",

    "comparison_rules_frozen_from_branch_A":
        True,

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform()
}


validation_status = (
    "Completed without discrepancies"
    if (
        fully_correct_count
        == reference_count
        and missing_count == 0
        and unsupported_count == 0
        and schema_validity
    )
    else
    "Completed with discrepancies"
)


VALIDATION_CONCLUSION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "validation_status":
        validation_status,

    "reference_records":
        int(reference_count),

    "extracted_records":
        int(extracted_count),

    "aligned_records":
        int(aligned_count),

    "fully_correct_records":
        int(fully_correct_count),

    "discrepant_records":
        int(discrepant_count),

    "missing_records":
        int(missing_count),

    "unsupported_extracted_records":
        int(unsupported_count),

    "completeness":
        completeness,

    "record_precision_exact":
        record_precision_exact,

    "record_recall_exact":
        record_recall_exact,

    "record_f1_exact":
        record_f1_exact,

    "overall_primary_field_accuracy":
        overall_primary_field_accuracy,

    "schema_valid":
        bool(schema_validity),

    "normalisation_integrity_passed":
        bool(normalisation_integrity_passed),

    "parent_B_equivalence_passed":
        bool(parent_equivalence_passed),

    "comparison_rules_frozen_from_branch_A":
        True
}


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

detailed_df.to_csv(
    PATHS["detailed"],
    index=False,
    encoding="utf-8-sig"
)

fully_correct_df.to_csv(
    PATHS["fully_correct"],
    index=False,
    encoding="utf-8-sig"
)

discrepant_df.to_csv(
    PATHS["discrepant"],
    index=False,
    encoding="utf-8-sig"
)

missing_df.to_csv(
    PATHS["missing"],
    index=False,
    encoding="utf-8-sig"
)

unsupported_df.to_csv(
    PATHS["unsupported"],
    index=False,
    encoding="utf-8-sig"
)

field_validation_df.to_csv(
    PATHS["field_validation"],
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    PATHS["category_metrics"],
    index=False,
    encoding="utf-8-sig"
)

year_metrics_df.to_csv(
    PATHS["year_metrics"],
    index=False,
    encoding="utf-8-sig"
)

PATHS["summary"].write_text(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

PATHS["metadata"].write_text(
    json.dumps(
        VALIDATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

PATHS["conclusion"].write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final methodological/provenance assertions
# ------------------------------------------------------------

if not reference_semantics_valid:
    raise AssertionError(
        "D13 fixed Stage 1 reference semantic validation failed."
    )

if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "Parsed extraction hash does not match Branch C metadata."
    )

if not source_hash_matches_stage_1:
    raise AssertionError(
        "Branch C source hash does not match the frozen Stage 1 source."
    )

if not metadata_document_id_correct:
    raise AssertionError(
        "Branch C metadata document_id is incorrect."
    )

if not metadata_branch_correct:
    raise AssertionError(
        "Branch C metadata branch is incorrect."
    )

if not normalisation_document_id_correct:
    raise AssertionError(
        "Branch C normalisation artefact document_id is incorrect."
    )

if not normalisation_branch_correct:
    raise AssertionError(
        "Branch C normalisation artefact branch is incorrect."
    )

if not normalisation_parent_branch_correct:
    raise AssertionError(
        "Branch C normalisation artefact parent branch is incorrect."
    )

if not parent_equivalence_passed:
    raise AssertionError(
        "Branch C parent-B equivalence verification failed."
    )

if not normalisation_integrity_passed:
    raise AssertionError(
        "Branch C normalisation-integrity verification failed."
    )

if not records_is_list:
    raise AssertionError(
        "Branch C extraction does not contain an evaluable records list."
    )


# Accounting checks validate the validator, not model quality.
assert (
    aligned_count
    + missing_count
    == reference_count
)

assert (
    aligned_count
    + unsupported_count
    == extracted_count
)

assert (
    fully_correct_count
    + discrepant_count
    == aligned_count
)


required_outputs = list(
    PATHS.values()
)

missing_outputs = [
    path.name
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing output files: {missing_outputs}"
    )


print("Validation C — D13 completed successfully.")
print("Validation status:", validation_status)
print("Reference records:", reference_count)
print("Extracted records:", extracted_count)
print("Aligned records:", aligned_count)
print("Fully correct records:", fully_correct_count)
print("Discrepant records:", discrepant_count)
print("Missing records:", missing_count)
print("Unsupported/unmatched records:", unsupported_count)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Primary field accuracy:",
    None
    if overall_primary_field_accuracy is None
    else round(
        overall_primary_field_accuracy,
        4
    )
)
print("Schema validity:", schema_validity)
print(
    "Normalisation integrity passed:",
    normalisation_integrity_passed
)
print(
    "Comparison rules frozen from Branch A:",
    True
)

print("\nGenerated files:")
for path in required_outputs:
    print("-", path.name)

for path in required_outputs:
    files.download(path)

Validation C — D13 completed successfully.
Validation status: Completed without discrepancies
Reference records: 75
Extracted records: 75
Aligned records: 75
Fully correct records: 75
Discrepant records: 0
Missing records: 0
Unsupported/unmatched records: 0
Completeness: 1.0
Exact F1: 1.0
Primary field accuracy: 1.0
Schema validity: True
Normalisation integrity passed: True
Comparison rules frozen from Branch A: True

Generated files:
- D13_branch_C_validation_detailed.csv
- D13_branch_C_fully_correct_records.csv
- D13_branch_C_discrepant_records.csv
- D13_branch_C_missing_records.csv
- D13_branch_C_unsupported_records.csv
- D13_branch_C_field_validation.csv
- D13_branch_C_category_metrics.csv
- D13_branch_C_year_metrics.csv
- D13_branch_C_alignment_issues.json
- D13_reference_semantics_confirmation.json
- D13_branch_C_validation_summary.json
- D13_branch_C_validation_metadata.json
- D13_branch_C_validation_conclusion.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>